# Climate Indices with Earthkit & Xclim




In [1]:
import xarray as xr
import earthkit.data as ekd
import earthkit.plots as ekp
from earthkit.climate.indicators.temperature import (
    daily_temperature_range,
    heating_degree_days,
    warm_spell_duration_index,
)
from earthkit.climate.utils.percentile import calculate_percentile_doy

ekd.settings.set("cache-policy", "user")


## Loading CMIP6 data

We’ll use *daily gridded data* from the ACCESS-CM2 model for precipitation (`pr`), maximum (`tasmax`) and minimum (`tasmin`) temperature, for both historical and SSP585 future scenarios.


In [2]:
# Load precipitation
pr_hist = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/pr_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
)
pr_ssp585 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/pr_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
)

# Load temperature
tasmin_hist = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmin_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
)
tasmin_ssp585 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmin_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
)

tasmax_hist = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmax_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
)
tasmax_ssp585 = ekd.from_source(
    "url",
    "https://sites.ecmwf.int/repository/earthkit-climate/tasmax_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
)

In [3]:
tasmax_hist_k = tasmax_hist.to_xarray() + 273.15
tasmax_hist_k["tasmax"].attrs["units"] = "K"
tasmin_hist_k = tasmin_hist.to_xarray() + 273.15
tasmin_hist_k["tasmin"].attrs["units"] = "K"

ds_hist_k = xr.merge([tasmax_hist_k, tasmin_hist_k])
ds_hist_k

/var/folders/l2/529q7bzs665bnrn7_wjx1nsr0000gn/T/ipykernel_60683/4161819160.py:6: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_hist_k = xr.merge([tasmax_hist_k, tasmin_hist_k])


<xarray.Dataset> Size: 236MB
Dimensions:  (lat: 48, lon: 84, time: 7305)
Coordinates:
  * lat      (lat) float64 384B 41.62 41.67 41.72 41.77 ... 43.87 43.92 43.97
  * lon      (lon) float64 672B -9.575 -9.525 -9.475 ... -5.525 -5.475 -5.425
  * time     (time) datetime64[ns] 58kB 1995-01-01 1995-01-02 ... 2014-12-31
    height   float64 8B 2.0
Data variables:
    tasmax   (time, lat, lon) float32 118MB dask.array<chunksize=(3653, 24, 42), meta=np.ndarray>
    tasmin   (time, lat, lon) float32 118MB dask.array<chunksize=(3653, 24, 42), meta=np.ndarray>
Attributes:
    model:         ACCESS-CM2_r1i1p1f1_deepESD
    scenario:      historical
    project_name:  cmip6
    project_type:  projections

In [4]:
# DTR
dtr = daily_temperature_range(ds_hist_k)
dtr


0 {}
1 {}


/opt/homebrew/Caskroom/miniforge/base/envs/ek-hack/lib/python3.12/site-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/opt/homebrew/Caskroom/miniforge/base/envs/ek-hack/lib/python3.12/site-packages/xclim/core/cfchecks.py:79: UserWarning: Variable does not have a `standard_name` attribute.
  check_valid(vardata, "standard_name", data["standard_name"])


<xarray.DataArray 'dtr' (time: 20, lat: 48, lon: 84)> Size: 323kB
dask.array<where, shape=(20, 48, 84), dtype=float32, chunksize=(1, 24, 42), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 160B 1995-01-01 1996-01-01 ... 2014-01-01
  * lat      (lat) float64 384B 41.62 41.67 41.72 41.77 ... 43.87 43.92 43.97
  * lon      (lon) float64 672B -9.575 -9.525 -9.475 ... -5.525 -5.475 -5.425
    height   float64 8B 2.0
Attributes:
    units:              K
    units_metadata:     temperature: difference
    cell_methods:        time range within days time: mean over days
    history:            [2026-01-29 17:37:42] dtr: DTR(tasmin=tasmin, tasmax=...
    standard_name:      air_temperature
    long_name:          Mean diurnal temperature range
    description:        Annual mean diurnal temperature range.
    earthkit_metadata:  {'call_info': {'module': 'earthkit.climate.indicators...

In [5]:
try:
    dtr = dtr.to_xarray()
except Exception:
    pass
dtr.attrs

{'units': 'K',
 'units_metadata': 'temperature: difference',
 'cell_methods': ' time range within days time: mean over days',
 'history': "[2026-01-29 17:37:42] dtr: DTR(tasmin=tasmin, tasmax=tasmax, freq='YS') with options check_missing=any - xclim version: 0.60.0\ntasmin: \ntasmax: ",
 'standard_name': 'air_temperature',
 'long_name': 'Mean diurnal temperature range',
 'description': 'Annual mean diurnal temperature range.',
 'earthkit_metadata': {'call_info': {'module': 'earthkit.climate.indicators.temperature',
   'function': 'daily_temperature_range',
   'parameters': {'ds': xarray.core.dataset.Dataset, 'kwargs': dict}},
  'xclim_version': '0.60.0'}}